# RFSoC4x2 — General Waveform TX + RX

**Single interface** for tone, FMCW chirp, triangular chirp, pulse, or custom samples. Configure IF/RF center, bandwidth, duration and amplitude, then transmit/capture using the **already-installed AMD/Xilinx MTS overlay**.

**Important:** `carrier_hz` is a desired **RF frequency**, not necessarily something the DAC can physically generate. In `external_mixer` mode, Python computes an IF/LO plan; it does not program an external LO. Both DAC_A and DAC_B output the same waveform in this overlay. Stop other Jupyter kernels before loading hardware.

The first cells run entirely offline. **No RF output starts without your explicit enable.**

## 1. Editable signal parameters — change these every time you want

In [ ]:
from waveforms import RadarConfig, generate, describe
from mts_hardware import MTSHardware
from processing import spectrum_real, circular_delay
import numpy as np
import matplotlib.pyplot as plt

# --- ALL SIGNAL PARAMETERS ---
cfg = RadarConfig(
    waveform="chirp",        # "tone", "chirp", "triangle", "pulse", "custom"
    rf_mode="direct",         # "direct" (cable IF) or "external_mixer" (RF front end)
    carrier_hz=300e6,         # RF center: e.g. 300e6 loopback, 5.5e9 C-band
    bandwidth_hz=200e6,      # Chirp sweep bandwidth; 0 for "tone" or "pulse"
    chirp_duration_s=None,    # None = entire DAC RAM period; e.g. 2e-6 if it fits
    amplitude=0.15,           # 0 ... 1, relative to DAC full scale
    phase_rad=0.0,
    if_center_hz=300e6,       # Used only for "external_mixer" (e.g. IF 300 MHz)
    sideband="upper",        # "upper" -> LO=RF-IF; "lower" -> LO=RF+IF
    custom_file=None,        # e.g. "my_samples.npy", for waveform="custom"
    custom_format="dac_samples", # "dac_samples" or "complex_envelope"
)

# ALTERNATIVE: load an entire configuration from a JSON file:
# cfg = RadarConfig.read_json("config_loopback.json")
# cfg = RadarConfig.read_json("config_cband.json")

# --- HARDWARE SWITCHES: initially all disabled ---
CONNECT_HARDWARE = False    # Set True to program the FPGA with the MTS bitstream
ENABLE_TRANSMISSION = False  # Set True ONLY after checking RF signal levels and connectors
CAPTURE_RX = False          # Starts/retriggers DAC playback during capture
CAPTURE_N = 8192


## 2. Preview and validate waveform (no board required)

The preview uses a **placeholder RAM length** because only the running overlay can report the real DAC memory size. Once hardware is loaded we regenerate the waveform for its actual length.

In [ ]:
PREVIEW_RAM_SAMPLES = 16384
preview, preview_plan = generate(cfg, PREVIEW_RAM_SAMPLES, base_directory=".")
print(describe(cfg, preview_plan, PREVIEW_RAM_SAMPLES))

fig, ax = plt.subplots(2, 1, figsize=(11, 6))
t = np.arange(preview.size) / cfg.sample_rate_hz
ax[0].plot(1e6 * t[:min(4000, preview.size)], preview[:min(4000, preview.size)])
ax[0].set(xlabel="Time [us]", ylabel="Normalized DAC amplitude")
ax[0].grid(True)
f, db = spectrum_real(preview, cfg.sample_rate_hz)
ax[1].plot(f / 1e6, db)
ax[1].set(xlabel="Frequency [MHz]", ylabel="Normalized spectrum [dB]", ylim=(-100, 5))
ax[1].set_xlim(0, min(1000, cfg.sample_rate_hz / 2e6))
ax[1].grid(True)
plt.tight_layout()
plt.show()


## 3. Connect to the real RFSoC

Run this only inside the RFSoC4x2's PYNQ notebook environment, outside the AMD source checkout. Shut down all other kernels first. `load()` programs the FPGA, but **leaves TX off**.

In [ ]:
hw = None
if CONNECT_HARDWARE:
    hw = MTSHardware()
    hw.load()
else:
    print("Simulation/preview only. Set CONNECT_HARDWARE=True to load MTS on RFSoC4x2.")


## 4. Generate and write exactly one DAC RAM period

The sample count comes from the hardware. If the requested `chirp_duration_s` is too long, this cell raises a descriptive error. **Writing the RAM stops TX**, so you can change parameters and rerun safely.

In [ ]:
tx_samples = None
if hw is not None:
    tx_samples, actual_plan = generate(cfg, hw.ram_samples, base_directory=".")
    print(describe(cfg, actual_plan, hw.ram_samples))
    codes = hw.write(tx_samples)
else:
    print("No hardware connected; TX waveform not written.")


## 5. Transmit — requires a safe RF connection

For **loopback**, use known-safe attenuators and DC blocks as specified by your DAC/ADC board docs; verify signal levels. The reference MTS notebook uses DAC_A → ADC_B, but the signal plays on **both DAC_A and DAC_B**.

For **external C-band RF**, you must configure the LO, mixers, filters and any power amplifier yourself. Software does not configure that front end. Check emissions authorization before transmitting in air.

In [ ]:
if hw is not None and ENABLE_TRANSMISSION:
    hw.start(confirm_hardware_ready=True)
else:
    print("DAC transmission is OFF (set CONNECT_HARDWARE=True and ENABLE_TRANSMISSION=True).")


## 6. Optional ADC capture and spectrum

The MTS driver's `internal_capture()` **restarts DAC playback**. Each of the three returned rows is an internal ADC capture stream; the ordering is **not automatically the same as SMA connector names**. Confirm channel mapping with an injected tone. The frequency axis below assumes the nominal 4 GS/s and contiguous/reordered samples as appropriate; validate the actual RFDC capture configuration before treating it as calibrated.

In [ ]:
rx = None
if hw is not None and ENABLE_TRANSMISSION and CAPTURE_RX:
    rx = hw.capture_internal(CAPTURE_N, confirm_hardware_ready=True)
    print("Captured raw int16:", rx.shape)
    fig, ax = plt.subplots(2, 1, figsize=(11, 6))
    for i, channel in enumerate(rx):
        ax[0].plot(np.arange(min(1500, len(channel))) / cfg.sample_rate_hz * 1e6,
                   channel[:1500], label=f"Internal stream {i}", alpha=.75)
        f, s = spectrum_real(channel, cfg.sample_rate_hz)
        ax[1].plot(f/1e6, s, label=f"Internal stream {i}", alpha=.75)
    ax[0].set(xlabel="Nominal time [us]", ylabel="ADC code")
    ax[1].set(xlabel="Nominal frequency [MHz]", ylabel="Normalized magnitude [dB]", ylim=(-100, 5))
    ax[1].set_xlim(0, 1000)
    for a in ax:
        a.legend()
        a.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Capture disabled. Set CAPTURE_RX=True after you have connected a safe loopback.")


## 7. Optional relative delay diagnostic

Only use after verifying **which ADC stream contains your injected signal**, the capture sample rate/order, and equal-length records. Circular correlation returns a delay **modulo the RAM repetition period**, which includes unknown TX/RX chain latencies. This is **not yet a calibrated FMCW range estimate**.

In [ ]:
RX_STREAM_INDEX = 0  # CHANGE only after identifying the physical RX stream.
if rx is not None and tx_samples is not None and rx.shape[1] == len(tx_samples):
    lag, dt = circular_delay(tx_samples, rx[RX_STREAM_INDEX], cfg.sample_rate_hz)
    print(f"Diagnostic circular lag: {lag} samples, {dt * 1e9:.3f} ns")
else:
    print("Delay diagnostic skipped: need confirmed stream and one full RAM period of RX data.")


## 8. Optional experimental digital dechirping

For a real IF chirp, the code constructs analytic TX/RX signals and computes `beat = RX * conj(TX)`, then FFT. This **requires** matching sample time/order, one chirp period, the right RX stream, and compensated fixed TX/RX delay. **The raw MTS capture does not yet establish all those conditions**: use this only to explore data, not to claim absolute range. It is disabled by default.

In [ ]:
from processing import digital_dechirp, beat_peak_hz, range_from_beat
TRY_DIGITAL_DECHIRP = False
CABLE_VELOCITY_FACTOR = 0.66
if (TRY_DIGITAL_DECHIRP and rx is not None and tx_samples is not None
        and cfg.waveform == "chirp" and rx.shape[1] == len(tx_samples)):
    beat = digital_dechirp(tx_samples, rx[RX_STREAM_INDEX])
    f_beat_hz = beat_peak_hz(beat, cfg.sample_rate_hz)
    sweep_duration = len(tx_samples) / cfg.sample_rate_hz if cfg.chirp_duration_s is None else cfg.chirp_duration_s
    delay_s = abs(f_beat_hz) * sweep_duration / cfg.bandwidth_hz
    length_m = range_from_beat(f_beat_hz, cfg.bandwidth_hz, sweep_duration,
                               geometry="cable", vf=CABLE_VELOCITY_FACTOR)
    print(f"Dominant beat: {f_beat_hz / 1e3:+.2f} kHz")
    print(f"Raw total delay: {delay_s * 1e9:.2f} ns")
    print(f"Uncalibrated equivalent coax length: {length_m:.3f} m")
    print("WARNING: Do not treat these values as calibrated ranging until trigger, channel order and latency are verified.")
else:
    print("Digital dechirp not run; set TRY_DIGITAL_DECHIRP=True only after validating matching full-period RX data.")

## 9. Stop transmit

In [ ]:
if hw is not None:
    hw.stop()
    print("DAC playback disabled")
else:
    print("No RFSoC hardware active")
